In [1]:
from copy import copy
import os
import csv
from Bio import SeqIO
from collections import defaultdict

def read_smiles_file(smiles_file_path, binding_value):
    compound_data = []
    with open(smiles_file_path, 'r') as file:
        for line in file:
            compound_id, smiles = line.strip().split()
            compound_data.append((compound_id, smiles, binding_value))
    return compound_data

def main(fasta_file, targets_file, output_csv):
    # Read the sequences from the FASTA file and handle duplicates
    sequences = defaultdict(list)
    for record in SeqIO.parse(fasta_file, "fasta"):
        sequences[record.id].append(str(record.seq))

    with open(output_csv, mode='w', newline='') as csv_file:
        writer = csv.writer(csv_file)
        writer.writerow(["Protein ID", "Sequence", "SMILES", "Compound ID", "Binding"])

        with open(targets_file, 'r') as file:
            for line in file:
                protein_id, pdb_id = line.strip().split()

                # Get all sequences for the PDB ID
                all_sequences = sequences[pdb_id]

                ligands_file_path = os.path.join(protein_id, "ligands.smi")
                decoys_file_path = os.path.join(protein_id, "decoys.smi")
                chains = len(all_sequences)
                if os.path.exists(ligands_file_path):
                    ligands_data = read_smiles_file(ligands_file_path, binding_value=1)
                    
                    for i, sequence in enumerate(all_sequences):
                        if chains > 1:
                            protein_id_ = f"{protein_id}_chain_{i+1}"
                        else:
                            protein_id_ = copy(protein_id)
                        for compound_id, smiles, binding in ligands_data:
                            writer.writerow([protein_id, sequence, compound_id, smiles, binding])

                if os.path.exists(decoys_file_path):
                    decoys_data = read_smiles_file(decoys_file_path, binding_value=0)
                    for i, sequence in enumerate(all_sequences):
                        if chains > 1:
                            protein_id_ = f"{protein_id}_chain_{i+1}"
                        else:
                            protein_id_ = copy(protein_id)
                        for compound_id, smiles, binding in decoys_data:
                            writer.writerow([protein_id_, sequence, compound_id, smiles, binding])

main("merged_sequences.fasta", "LIST-PDB-ID.txt", "dude_z_dataset.csv")
